# Feature Engineering — team-perspective + Diff format

Reads the `matchups.csv` produced by your dataset builder (each game appears as 2 rows — one per team's view, with `Diff_*` features = `team − opp`) and adds new leakage-free features in the same style.

Output: **`matchups_v2.csv`** with all original columns plus new ones.

### Features added

**Diffs (this team − opponent):**
- `Diff_Days_Rest`
- `Diff_Win_Streak`
- `Diff_Last5_Win_Pct`, `Diff_Last10_Win_Pct`

**Per-team (this team's perspective):**
- `Team_Is_B2B`, `Opp_Is_B2B` — back-to-back flags for both sides
- `H2H_Win_Pct` — this team's win pct vs `OppID` in prior matchups this season
- `Elo_Win_Prob` — implied win probability from `Diff_Pregame_Elo` with home-court bump

All features use only games **strictly before** each game (in the same season).


## Setup

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

ELO_HOME_ADV = 100.0   # match the constant your dataset builder used

OUT_DIR = Path(".")
mm = pd.read_csv("matchups.csv", parse_dates=['DayDate'])
print(f"matchups.csv: {mm.shape}")
print(f"Columns ({len(mm.columns)}):")
print(list(mm.columns))

matchups.csv: (11969, 83)
Columns (83):
['T1_TeamID', 'T2_TeamID', 'Season', 'DayDate', 'GameID', 'ID', 'T1_Avg_Off_Eff', 'T1_Avg_Def_Eff', 'T1_Net_Rating', 'T1_Win_Pct', 'T1_Pregame_Elo', 'T1_Overall_Win_Pct', 'T1_Home_Win_Pct', 'T1_Away_Win_Pct', 'T1_Avg_Score', 'T1_Avg_FGM', 'T1_Avg_FGA', 'T1_Avg_FGM3', 'T1_Avg_FGA3', 'T1_Avg_FTM', 'T1_Avg_FTA', 'T1_Avg_OR', 'T1_Avg_DR', 'T1_Avg_Ast', 'T1_Avg_TO', 'T1_Avg_Stl', 'T1_Avg_Blk', 'T1_Avg_PF', 'T1_Avg_Opp_Score', 'T1_Avg_Opp_FGM', 'T1_Avg_Opp_FGA', 'T1_Avg_Opp_FGM3', 'T1_Avg_Opp_FGA3', 'T1_Avg_Opp_FTM', 'T1_Avg_Opp_FTA', 'T1_Avg_Opp_OR', 'T1_Avg_Opp_DR', 'T1_Avg_Opp_Ast', 'T1_Avg_Opp_TO', 'T1_Avg_Opp_Stl', 'T1_Avg_Opp_Blk', 'T1_Avg_Opp_PF', 'T1_seed', 'T1_Last_14_Days_Win_Pct', 'T2_Avg_Off_Eff', 'T2_Avg_Def_Eff', 'T2_Net_Rating', 'T2_Win_Pct', 'T2_Pregame_Elo', 'T2_Overall_Win_Pct', 'T2_Home_Win_Pct', 'T2_Away_Win_Pct', 'T2_Avg_Score', 'T2_Avg_FGM', 'T2_Avg_FGA', 'T2_Avg_FGM3', 'T2_Avg_FGA3', 'T2_Avg_FTM', 'T2_Avg_FTA', 'T2_Avg_OR', 'T2_A

## Step 1 — Per-team chronological features

Sort by `(TeamID, Season, DayDate, GameID)` and walk each team's chronological log. Use `shift(1)` so the current game's outcome (`Target_Win`) never leaks into its own features.

In [2]:
def add_per_team_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.sort_values(['TeamID', 'Season', 'DayDate', 'GameID']).reset_index(drop=True).copy()
    grp_keys = ['TeamID', 'Season']

    # ---- Days_Rest: days since this team's previous game in same season ----
    prev_date = df.groupby(grp_keys)['DayDate'].shift(1)
    days_rest = (df['DayDate'] - prev_date).dt.days
    df['Days_Rest'] = days_rest.clip(upper=14)         # cap absurd gaps
    df['Is_B2B']    = (df['Days_Rest'] == 1).astype('Int64')
    df.loc[df['Days_Rest'].isna(), 'Is_B2B'] = pd.NA

    # ---- Win_Streak: +N for N-game win streak entering this game, -N for losses ----
    def streak(group):
        prev_won = group['Target_Win'].shift(1)
        s = 0; out = []
        for w in prev_won:
            if pd.isna(w):
                out.append(np.nan); s = 0
            elif w == 1:
                s = s + 1 if s >= 0 else 1
                out.append(s)
            else:
                s = s - 1 if s <= 0 else -1
                out.append(s)
        return pd.Series(out, index=group.index)

    df['Win_Streak'] = df.groupby(grp_keys, group_keys=False).apply(streak)

    # ---- Rolling N-game win pct (current game excluded via shift(1)) ----
    def rolling_winpct(n):
        return (df.groupby(grp_keys)['Target_Win']
                  .apply(lambda s: s.shift(1).rolling(n, min_periods=1).mean())
                  .reset_index(level=grp_keys, drop=True))

    df['Last5_Win_Pct']  = rolling_winpct(5)
    df['Last10_Win_Pct'] = rolling_winpct(10)

    return df


mm = add_per_team_features(mm)
mm[['Season','DayDate','TeamID','OppID','Target_Win',
    'Days_Rest','Is_B2B','Win_Streak',
    'Last5_Win_Pct','Last10_Win_Pct']].head(15)

KeyError: 'TeamID'

## Step 2 — Self-join to get opponent's per-team features, then diff

Each row's "opponent values" live in a *different* row of the same DataFrame (the opposite team's perspective on the same game). We rebuild them via a self-join on `(GameID, OppID) → (GameID, TeamID)`, then compute `Diff_* = team − opp`.

In [ ]:
PER_TEAM = ['Days_Rest', 'Is_B2B', 'Win_Streak']

opp_lookup = (mm[['GameID', 'TeamID'] + PER_TEAM]
              .rename(columns={'TeamID': 'OppID',
                               **{c: f'Opp_{c}' for c in PER_TEAM}}))

mm = mm.merge(opp_lookup, on=['GameID', 'OppID'], how='left')

# Diff versions of the continuous per-team features
mm['Diff_Days_Rest']      = mm['Days_Rest']      - mm['Opp_Days_Rest']
mm['Diff_Win_Streak']     = mm['Win_Streak']     - mm['Opp_Win_Streak']
mm['Diff_Last5_Win_Pct']  = mm['Last5_Win_Pct']  - mm['Opp_Last5_Win_Pct']
mm['Diff_Last10_Win_Pct'] = mm['Last10_Win_Pct'] - mm['Opp_Last10_Win_Pct']

# B2B is binary -- keep both perspectives separately
mm = mm.rename(columns={'Is_B2B': 'Team_Is_B2B'})

# Drop the per-team intermediates (we keep the diffs and the B2B flags)
mm = mm.drop(columns=['Days_Rest', 'Win_Streak', 'Last5_Win_Pct', 'Last10_Win_Pct',
                      'Opp_Days_Rest', 'Opp_Win_Streak',
                      'Opp_Last5_Win_Pct', 'Opp_Last10_Win_Pct'])

mm[['TeamID','OppID','Team_Is_B2B','Opp_Is_B2B',
    'Diff_Days_Rest','Diff_Win_Streak',
    'Diff_Last5_Win_Pct','Diff_Last10_Win_Pct']].head(10)

## Step 3 — Head-to-head this season

Each row already has `TeamID` and `OppID`, so we can compute this team's prior win pct vs that specific opponent within the season directly. NaN means it's the first meeting this season.

In [ ]:
def add_h2h(df: pd.DataFrame) -> pd.DataFrame:
    df = df.sort_values(['Season','TeamID','OppID','DayDate','GameID']).reset_index(drop=True).copy()
    grp = df.groupby(['Season','TeamID','OppID'], sort=False)
    cum_wins  = grp['Target_Win'].cumsum() - df['Target_Win']   # wins BEFORE current
    cum_games = grp.cumcount()                                  # games BEFORE current
    df['H2H_Win_Pct'] = cum_wins / cum_games.replace(0, np.nan)
    return df


mm = add_h2h(mm)
mm[['Season','DayDate','TeamID','OppID','Target_Win','H2H_Win_Pct']].head(15)

## Step 4 — Elo-implied win probability

`Diff_Pregame_Elo` already encodes `team_elo − opp_elo` from this team's view. Add the home-court bump (positive for the home team, negative for the away team) and convert with the standard logistic:

$$P(\text{this team wins}) = \frac{1}{1 + 10^{-(\text{Diff\_Pregame\_Elo}\,+\,\text{HCA}_{\text{signed}}) / 400}}$$

where $\text{HCA}_{\text{signed}}$ is +100 if `Is_Home` and −100 otherwise.

In [ ]:
hca = np.where(mm['Is_Home'].astype(bool), ELO_HOME_ADV, -ELO_HOME_ADV)
mm['Elo_Win_Prob'] = 1.0 / (1.0 + 10 ** (-((mm['Diff_Pregame_Elo'] + hca)) / 400.0))

print(f"Elo_Win_Prob range: [{mm['Elo_Win_Prob'].min():.3f}, "
      f"{mm['Elo_Win_Prob'].max():.3f}]")
print(f"Home rows mean: {mm.loc[mm['Is_Home']==1, 'Elo_Win_Prob'].mean():.3f}  "
      f"(should be > 0.5)")
print(f"Away rows mean: {mm.loc[mm['Is_Home']==0, 'Elo_Win_Prob'].mean():.3f}  "
      f"(should be < 0.5)")

## Step 5 — Save `matchups_v2.csv`

In [ ]:
mm = mm.sort_values(['Season','DayDate','GameID','TeamID']).reset_index(drop=True)

mm.to_csv(OUT_DIR / "matchups_v2.csv", index=False)
print(f"Wrote matchups_v2.csv  ({len(mm):,} rows, {len(mm.columns)} cols)")

new_cols = ['Team_Is_B2B','Opp_Is_B2B',
            'Diff_Days_Rest','Diff_Win_Streak',
            'Diff_Last5_Win_Pct','Diff_Last10_Win_Pct',
            'H2H_Win_Pct','Elo_Win_Prob']
print(f"\nNew features added ({len(new_cols)}):")
for c in new_cols:
    null_count = mm[c].isna().sum()
    print(f"  {c:<24s}  (NaN: {null_count})")

## Sanity checks

The team-perspective format gives us a strong consistency check: for any feature that's symmetric across the two perspectives of a game, the sum across both rows should equal a known constant.

- `Diff_*` features → **sum = 0** across both perspectives (because team's diff = −opp's diff)
- `Elo_Win_Prob` → **sum = 1** (one team's win prob + the other's = 1)
- `H2H_Win_Pct` → sum = 1 when both perspectives are non-NaN (they share the same prior matchup history)

In [ ]:
# Check 1: Last5_Win_Pct correctness via independent reconstruction
prior_count = mm.groupby(['TeamID','Season']).cumcount()
sample_idx = mm[(prior_count >= 5) & mm['Diff_Last5_Win_Pct'].notna()].index[0]
sample = mm.loc[sample_idx]

team_priors = mm[(mm['TeamID']==sample['TeamID']) & (mm['Season']==sample['Season']) &
                 (mm['DayDate']<sample['DayDate'])].sort_values('DayDate').tail(5)
opp_priors  = mm[(mm['TeamID']==sample['OppID'])  & (mm['Season']==sample['Season']) &
                 (mm['DayDate']<sample['DayDate'])].sort_values('DayDate').tail(5)
diff_manual = team_priors['Target_Win'].mean() - opp_priors['Target_Win'].mean()

print(f"Sample team={sample['TeamID']}, opp={sample['OppID']}, "
      f"season={sample['Season']}, date={sample['DayDate'].date()}")
print(f"  Team last5 win pct: {team_priors['Target_Win'].mean():.3f}")
print(f"  Opp  last5 win pct: {opp_priors['Target_Win'].mean():.3f}")
print(f"  Manual diff:        {diff_manual:.3f}")
print(f"  File Diff_Last5:    {sample['Diff_Last5_Win_Pct']:.3f}")
print(f"  Match: {np.isclose(diff_manual, sample['Diff_Last5_Win_Pct'])}")

# Check 2: Diff_* should sum to 0 across both perspectives of the same game
print("\n=== Diff sums per game (expected ~0) ===")
for col in ['Diff_Days_Rest','Diff_Win_Streak',
            'Diff_Last5_Win_Pct','Diff_Last10_Win_Pct']:
    s = mm.groupby('GameID')[col].sum(min_count=2)   # NaN if either side is NaN
    print(f"  {col:<22s}  mean abs sum: {s.abs().mean():.6f}")

# Check 3: Elo_Win_Prob should sum to 1.0 across both perspectives
prob_sum = mm.groupby('GameID')['Elo_Win_Prob'].sum()
print(f"\nElo_Win_Prob per-game sum: mean={prob_sum.mean():.6f}, "
      f"std={prob_sum.std():.2e}  (expect 1.0, ~0)")

# Check 4: H2H_Win_Pct should sum to 1.0 when both perspectives non-NaN
h2h_sum = mm.groupby('GameID')['H2H_Win_Pct'].sum(min_count=2)
nonnan = h2h_sum.dropna()
print(f"H2H_Win_Pct per-game sum (when both non-NaN): "
      f"mean={nonnan.mean():.6f}, n={len(nonnan)}  (expect 1.0)")

## Plug into your model

When you load `matchups_v2.csv`, identify the feature columns dynamically (any `Diff_*` plus the per-team binary/probability features). Drop the rows from each team's first ~1 game of the season where rolling features are NaN — XGBoost can handle NaN, but with only 38 train features and dataset already double-counted, it doesn't hurt to be clean.

```python
df = pd.read_csv("matchups_v2.csv", parse_dates=['DayDate'])
df = df.dropna(subset=['Diff_Avg_Score']).copy()

feature_cols = [c for c in df.columns if c.startswith('Diff_')] + [
    'Is_Home', 'Team_Is_B2B', 'Opp_Is_B2B',
    'H2H_Win_Pct', 'Elo_Win_Prob',
]
X = df[feature_cols]
y = df['Target_Win']
```

Then your existing Leave-One-Season-Out CV loop works unchanged — the same-game's two rows are always in the same season, so there's no train/val leak.